# 1. Set up

## 1.1 Kernel

- For training users, please select the kernel "training"
- After choosing your kernel, wait for a few minutes until the kernel initializes properly

## 1.2 Spark Session
A spark session named ```spark``` is already built for you based on the configuration of your chosen template

The AppName is the same name in the Ocean Spark Dashboard. It is good practice to monitor your notebook in the dashboard. The notebook status should be "Running" (blue circle)

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

# 2. Import packages

In [2]:
import os
import pandas as pd
from datetime import datetime

In [3]:
from ais import functions as af

In [4]:
import pyspark.sql.functions as F

In [5]:
import h3
import folium
from shapely.geometry import Polygon

# 3. Read AIS Data

AIS data and ship register data are stored in Amazon S3 buckets.
We prepared a sample dataset for training users, which is a subset of the AIS data.
The training dataset contains AIS records of January 2024.

There are two ways to read the AIS data.
- Set a basepath, and read the AIS data using spark.read()
- Use the ais package to read the AIS data

## 3.1 Use spark.read() 

import os
basepath = os.environ.get(
    "AIS_BASEPATH",
    "s3a://ais-data-142496269814/exact-earth-data/transformed/prod/"
)
basepath- Get the base path

In [6]:
import os
basepath = os.environ.get(
    "AIS_BASEPATH",
    "s3a://ais-data-142496269814/exact-earth-data/transformed/prod/"
)
basepath

's3a://ais-data-142496269814/exact-earth-data/transformed/prod/'

- Read ALL AIS for one date 

Read from path ```basepath + "year=<YYYY>/month=<mm>/day=<dd>"```

In [7]:
df1 = spark.read.option("basepath",basepath).parquet(basepath+"year=2024/month=01/day=01")

In [8]:
df1.limit(1).show(vertical=True, truncate=True)

-RECORD 0---------------------------------
 mmsi              | 259511000            
 message_type      | 1                    
 dt_insert_utc     | 2024-01-01 15:52:08  
 longitude         | 32.74504667          
 latitude          | 76.37381667          
 imo               | 8131453              
 vessel_name       | VIMA                 
 callsign          | LJBD                 
 vessel_type       | Fishing              
 vessel_type_code  | 30                   
 vessel_type_cargo | NULL                 
 vessel_class      | A                    
 length            | 78.0                 
 width             | 12.0                 
 flag_country      | Norway               
 flag_code         | 259                  
 destination       | TROMSO               
 eta               | 3211234              
 draught           | 0.0                  
 sog               | 6.7                  
 cog               | 181.9                
 rot               | 0.0                  
 heading   

In [9]:
#Cara 1: Show (paling umum)
df1.show(5)  # tampilkan 5 baris

+---------+------------+-------------------+-----------+-----------+-------+-----------+--------+-----------+----------------+-----------------+------------+------+-----+------------+---------+-----------+-------+-------+---+-----+---+-------+----------------+---------------+------+-------------------+-------------------+----------------+---------------+-------------------+--------------------+---------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+-----+---+
|     mmsi|message_type|      dt_insert_utc|  longitude|   latitude|    imo|vessel_name|callsign|vessel_type|vessel_type_code|vessel_type_cargo|vessel_class|length|width|flag_country|flag_code|destination|    eta|draught|sog|  cog|rot|heading|      nav_status|nav_sta

In [11]:
#Cara 2: Show dengan kolom tidak terpotong
df1.show(5, truncate=False)

+---------+------------+-------------------+-----------+-----------+-------+-----------+--------+-----------+----------------+-----------------+------------+------+-----+------------+---------+-----------+-------+-------+---+-----+---+-------+----------------+---------------+------+-------------------+-------------------+----------------+---------------+-------------------+-----------------------------------------------------------------------------------------------------------+---------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----+-----+---+
|mmsi     |message_type|dt_insert_utc      |longitude  |latitude   |imo    |vessel_name|callsign|vessel_type|vessel_type_code|vessel_type_cargo|vessel_class|length|width|flag_count

- Read AIS data of a list of dates

In [12]:
#Cara 3: Print schema dulu
df1.printSchema()  # lihat struktur kolom
df1.show(5)        # lalu lihat datanya

root
 |-- mmsi: integer (nullable = true)
 |-- message_type: integer (nullable = true)
 |-- dt_insert_utc: timestamp (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- imo: integer (nullable = true)
 |-- vessel_name: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- vessel_type: string (nullable = true)
 |-- vessel_type_code: integer (nullable = true)
 |-- vessel_type_cargo: string (nullable = true)
 |-- vessel_class: string (nullable = true)
 |-- length: double (nullable = true)
 |-- width: double (nullable = true)
 |-- flag_country: string (nullable = true)
 |-- flag_code: integer (nullable = true)
 |-- destination: string (nullable = true)
 |-- eta: integer (nullable = true)
 |-- draught: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- cog: double (nullable = true)
 |-- rot: double (nullable = true)
 |-- heading: double (nullable = true)
 |-- nav_status: string (nullable = true)
 |-- nav_status

In [13]:
#Cara 4: Konversi ke Pandas (lebih rapi)
df1.limit(10).toPandas()  # tampilkan 10 baris dalam format tabel

,mmsi,message_type,dt_insert_utc,longitude,latitude,imo,vessel_name,callsign,vessel_type,vessel_type_code,...,H3_int_index_9,H3_int_index_10,H3_int_index_11,H3_int_index_12,H3_int_index_13,H3_int_index_14,H3_int_index_15,year,month,day
0,257757000,1,2024-01-01 00:01:51,33.216807,76.425993,8709896,PROWESS,LCUF,Fishing,30,...,616995763115524095,621499362742730751,626002962370097151,630506561997464063,635010161624834431,639513761252206815,644017360879577307,2024,1,1
1,257757000,1,2024-01-01 00:34:43,33.249632,76.401180,8709896,PROWESS,LCUF,Fishing,30,...,616995763210944511,621499362838151167,626002962465513471,630506562092880895,635010161720250943,639513761347621423,644017360974991913,2024,1,1
2,257757000,1,2024-01-01 01:28:06,33.287330,76.404163,8709896,PROWESS,LCUF,Fishing,30,...,616995762533040127,621499362160312319,626002961787678719,630506561415046143,635010161042416383,639513760669786871,644017360297157361,2024,1,1
3,257757000,1,2024-01-01 01:06:36,33.202852,76.371545,8709896,PROWESS,LCUF,Fishing,30,...,616995763064668159,621499362691809279,626002962319163391,630506561946530815,635010161573901119,639513761201271575,644017360828642068,2024,1,1
4,257757000,1,2024-01-01 00:53:10,33.180543,76.374932,8709896,PROWESS,LCUF,Fishing,30,...,616995763000705023,621499362627977215,626002962255339519,630506561882706431,635010161510076607,639513761137447079,644017360764817573,2024,1,1
5,257757000,3,2024-01-01 01:43:46,33.380318,76.421587,8709896,PROWESS,LCUF,Fishing,30,...,616995762459639807,621499362086944767,626002961714290687,630506561341657599,635010160969028031,639513760596398487,644017360223768976,2024,1,1
6,257757000,3,2024-01-01 01:30:11,33.298645,76.408275,8709896,PROWESS,LCUF,Fishing,30,...,616995762531991551,621499362159198207,626002961786552319,630506561413919743,635010161041289983,639513760668660447,644017360296030940,2024,1,1
7,257757000,3,2024-01-01 00:16:21,33.307325,76.427868,8709896,PROWESS,LCUF,Fishing,30,...,616995762466193407,621499362093367295,626002961720709119,630506561348076031,635010160975446207,639513760602816647,644017360230187178,2024,1,1
8,259511000,1,2024-01-01 00:52:54,33.338528,76.445872,8131453,VIMA,LJBD,Fishing,30,...,616995762684297215,621499362311438335,626002961938804735,630506561566172159,635010161193542399,639513760820912871,644017360448283364,2024,1,1
9,259511000,1,2024-01-01 01:06:36,33.403133,76.450310,8131453,VIMA,LJBD,Fishing,30,...,616995762678792191,621499362306064383,626002961933418495,630506561560785919,635010161188156095,639513760815526543,644017360442897036,2024,1,1


In [15]:
# Ganti 10 dengan jumlah yang diinginkan
df1_pd = df1.limit(1000).toPandas()
df1_pd.to_csv('hasil_ais.csv', index=False)

In [ ]:
#Read for one-week
#puth all date paths in a list
df_week = spark.read.parquet(*[basepath + "year=2024/month=01/day=01",
                          basepath + "year=2024/month=01/day=02",
                          basepath + "year=2024/month=01/day=03",
                          basepath + "year=2024/month=01/day=04",
                          basepath + "year=2024/month=01/day=05",
                          basepath + "year=2024/month=01/day=06",
                          basepath + "year=2024/month=01/day=07"]
                       )

In [ ]:
#Create a date column and then count rows per date
df_week = df_week.withColumn('date', F.to_date("dt_insert_utc"))
df_week.groupBy('date').count().show()

## 3.2 Use ais package

- Read AIS data of a certain day

In [ ]:
start_date = datetime.fromisoformat("2024-01-01")
df2 = af.get_ais(spark,start_date)

In [ ]:
df2.limit(1).show(vertical=True, truncate=True)

- read AIS data of a date range

NOTE: For complex calculations, it is recommended to read only maximum of 1 month of data.

In [ ]:
start_date = datetime.fromisoformat("2024-01-01")
end_date = datetime.fromisoformat("2024-01-05")
df3 = af.get_ais(spark,start_date, end_date = end_date)

In [ ]:
df3.withColumn("date",F.col("dt_insert_utc").cast("date")) \
        .groupby('date')  \
        .agg(F.count("eeid").alias("count")) \
        .orderBy("date") \
.show()

- read AIS data of a list of dates

In [ ]:
date_list = [datetime.fromisoformat("2024-01-01"),
             datetime.fromisoformat("2024-01-08"),
             datetime.fromisoformat("2024-01-15")]
date_list

In [ ]:
df4 = af.get_ais(spark,date_list=date_list)

In [ ]:
df4.withColumn("date",F.col("dt_insert_utc").cast("date")) \
        .groupby('date')  \
        .agg(F.count("eeid").alias("count")) \
        .orderBy("date") \
.show()

# 4. How to get AIS data of a certain place

## 4.1 Get H3 Index

We can get the H3 index of any pair of latitude and longitude using geo_to_h3 function from h3 package.

In [ ]:
#Eg:point in Suez Canal
latitude = 30.56529681740963
longitude = 32.33832930701776

In [ ]:
#Resolution of H3 Index used in partitioning
h3index_0 = h3.geo_to_h3(latitude, longitude, 0) #resolution 0
print(h3index_0)
h3index_1 = h3.geo_to_h3(latitude, longitude, 1) #resolution 1
print(h3index_1)
h3index_2 = h3.geo_to_h3(latitude, longitude, 2) #resolution 2
print(h3index_2)

 - We can visualize this using **folium** package. From the map, we see that Suez Canal is contained within H3 '803ffffffffffff'

In [ ]:
m = folium.Map(location=[latitude, longitude], zoom_start=4, tiles="cartodbpositron")
folium.Marker([latitude, longitude]).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_0, geo_json=True))).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_1, geo_json=True))).add_to(m)
folium.GeoJson(Polygon(h3.h3_to_geo_boundary(h3index_2, geo_json=True))).add_to(m)
m

# 4.2 filter data

Get the AIS records in side the H3 geo boundary

 - Resolution 0: 803ffffffffffff

In [ ]:
#If we want to read all the records within this area. 
#We can use the spark.sql(), and cast the H3 index.
df1.createOrReplaceTempView("temp_df")
df_H3_0 = spark.sql("""
                select dt_pos_utc, mmsi, longitude, latitude, H3index_0, H3_int_index_0, H3_int_index_1, H3_int_index_2, H3_int_index_3
                from temp_df
                where H3index_0 = "803ffffffffffff"
                """)

In [ ]:
df_H3_0.count()

- Resolution 1: 813e7ffffffffff

In [ ]:
h3index_1_int = int(h3index_1,16)
h3index_1_int

In [ ]:
df1.createOrReplaceTempView("temp_df")
df_H3_1 = spark.sql("""
                select dt_pos_utc, mmsi, longitude, latitude, H3index_0, H3_int_index_0, H3_int_index_1, H3_int_index_2, H3_int_index_3
                from temp_df
                where H3_int_index_1 = 582063863558569983
                """)

In [ ]:
df_H3_1.count()

# 5. Save and Download Data

## 5.1 Transform to DataFrame

In [ ]:
dataframe_H3_1 = df_H3_1.toPandas()
dataframe_H3_1.head(10)

## 5.2 Download Data

You can use create_download_link() function to download data. But please note that the load is limited. Do not try to download large dataset.

In [ ]:
af.create_download_link(dataframe_H3_1.head(100), title = "Download CSV file", filename = "SampleResult.csv")

# 6.  Map for AIS data

In [ ]:
import folium
from folium.plugins import HeatMap
from shapely.geometry import Polygon, MultiPoint, mapping
import h3

In [ ]:
def create_geojson (h3_list):
    geojson = {"type": 'FeatureCollection',
               "features": [{ "type": "Feature",
                            "geometry": {"type":"Polygon",
                                         "coordinates": (h3.h3_to_geo_boundary(x,geo_json=True),)
                                        },
                            "properties": {"H3 index":x,"Resolution":h3.h3_get_resolution(x)}, 
                           } for x in h3_list]
               }
    return geojson

In [ ]:
#In this example we choose the records near Suez Canal on Jan 1st 2024 as an example
#transform to Pandas format
df = df_H3_0.toPandas()
df

# 6.1 Heat map

In [ ]:
m = folium.Map(location=[latitude, longitude], zoom_start=6, tiles="cartodbpositron")
a = HeatMap(df[['latitude','longitude']], radius=10, blur=10, name="AIS Data HeatMap").add_to(m)
m

# 6.2 Map data with H3 resolutions

In [ ]:
df['h3_1'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],1), axis=1)
df['h3_2'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],2), axis=1)
df['h3_3'] = df[['latitude','longitude']].apply(lambda x: h3.geo_to_h3(x[0],x[1],3), axis=1)

In [ ]:
a = folium.GeoJson(create_geojson(df["h3_1"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 1', show=False).add_to(m)
a = folium.GeoJson(create_geojson(df["h3_2"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 2', show=False).add_to(m)
a = folium.GeoJson(create_geojson(df["h3_3"].unique().tolist()),
                    tooltip = folium.GeoJsonTooltip(fields=['H3 index','Resolution']),
                    name='H3 Resolution 3', show=False).add_to(m)
folium.LayerControl().add_to(m)
m

# 5. Stop Spark Session

Always end the spark session to free any executors assigned to your tasks, and shut down the kernel.

In [ ]:
spark.stop()